# Calibrar GWA contra estaciones de EEUU (ground-truth confiable, muestra grande)

Decisión de Pablo: en vez de seguir explicando San José/Finca Favorita como dos casos aislados
(Hallazgo 26/29), calibrar el patrón de error de GWA contra una muestra mucho más grande y con
ground-truth mucho más confiable -- la red ASOS de aeropuertos de EEUU, la mejor documentada del
mundo. La idea: si encontramos un patrón sistemático (ej. GWA falla más en terreno complejo/
boscoso, funciona bien en terreno abierto) con esta muestra grande, ese patrón debería generalizar
a cualquier país -- no solo arreglar Costa Rica.

### Aviso importante antes de arrancar -- una coordenada del catálogo NO es la coordenada real

Al armar la lista de estaciones se encontró que varias coordenadas del catálogo
(`datos_clima/epw_catalog_global.json`) para aeropuertos muy conocidos de EEUU están mal por
100-200+ km (Denver, Corpus Christi, Amarillo, Chicago O'Hare, Las Vegas, entre otras) -- ya se
sabía puntualmente para Finca Favorita (Hallazgo 18), pero acá se ve que es un problema más
extendido en el catálogo de lo documentado hasta ahora. **Por eso este notebook usa la coordenada
real que trae el propio encabezado de cada EPW** (`cargar_epw_real()` ya la expone en `meta['lat']`/
`meta['lon']`, más confiable que el catálogo) para todo el muestreo del ráster -- el catálogo solo
se usa para decidir QUÉ estación descargar, nunca para el punto exacto de muestreo.

### Segundo aviso -- tamaño del ráster de GWA para un país del tamaño de EEUU, sin confirmar

Costa Rica (~51,000 km²) se descargó sin problema. EEUU es ~163x más grande en área -- si GWA sirve
el ráster a resolución fija (250m nativo), el archivo podría ser mucho más grande. La celda de
descarga reporta el tamaño ANTES de intentar muestrear nada, para frenar temprano si resulta
impráctico en vez de descubrirlo a mitad de una descarga larga.

### Las 8 estaciones elegidas -- diversidad deliberada de terreno/cobertura

| Estación | Estado | Por qué |
|---|---|---|
| Dodge City Rgnl AP | KS | Llanura abierta, mínima complejidad -- caso "fácil" esperado |
| Corpus Christi Intl AP | TX | Costa abierta del Golfo, plana -- otro caso "fácil" |
| Phoenix Sky Harbor Intl AP | AZ | Desierto, vegetación mínima -- "fácil", régimen climático distinto |
| Reno Tahoe Intl AP | NV | Valle rodeado por Sierra Nevada -- análogo directo a San José |
| Denver Intl AP | CO | Llanura alta junto a las Rocosas -- complejidad moderada |
| Eugene AP (Mahlon Sweet Field) | OR | Valle de Willamette, cobertura forestal densa -- análogo a Finca Favorita |
| Monterey Rgnl AP | CA | Costa con terreno complejo (península, colinas junto al mar) |
| Chicago OHare Intl AP | IL | Urbano, junto a lago -- prueba de rugosidad urbana |

In [1]:
import os

def _find_repo_root():
    for candidato in ("..", "/content/ECO-Wind"):
        if os.path.exists(os.path.join(candidato, ".git")):
            return os.path.abspath(candidato)
    return None

repo = _find_repo_root()
if repo is None:
    repo = "/content/ECO-Wind"
    get_ipython().system(f"git clone https://github.com/Sogo2012/ECO-Wind.git {repo}")
else:
    get_ipython().system(f"git -C {repo} fetch origin main")
    get_ipython().system(f"git -C {repo} reset --hard origin/main")

get_ipython().run_line_magic("cd", f"{repo}/notebooks")
get_ipython().system(f"git -C {repo} log -1 --format='Commit activo: %h  %s  (%ci)'")

From https://github.com/Sogo2012/eco-wind
 * branch            main       -> FETCH_HEAD


HEAD is now at 46cbaea feat(fase2): notebook para calibrar GWA contra estaciones reales de EEUU


/home/user/eco-wind/notebooks
Commit activo: 46cbaea  feat(fase2): notebook para calibrar GWA contra estaciones reales de EEUU  (2026-09-01 01:17:30 +0000)


In [2]:
import sys
sys.path.insert(0, "..")

import json
import shutil

import numpy as np
import pandas as pd

from engine.epw_real import CARPETA_EPW_REAL, cargar_epw_real, descargar_y_extraer_epw
from engine.gwa_raster import descargar_raster_pais, muestrear_velocidad_media

pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Parte 1 — Descargar las 8 estaciones reales (EPW de climate.onebuilding.org)

In [3]:
NOMBRES_ESTACIONES = [
    "Dodge City Rgnl AP",
    "Corpus Christi Intl AP",
    "Phoenix Sky Harbor Intl AP",
    "Reno Tahoe Intl AP",
    "Denver Intl AP",
    "Eugene AP Sweet Field",
    "Monterey Rgnl AP",
    "Chicago OHare Intl AP",
]

catalogo = json.load(open(os.path.join(repo, "datos_clima", "epw_catalog_global.json")))
usa = catalogo["USA"]
por_nombre = {s["name"]: s for s in usa}

os.makedirs(CARPETA_EPW_REAL, exist_ok=True)
estaciones = []
for nombre in NOMBRES_ESTACIONES:
    if nombre not in por_nombre:
        print(f"=== {nombre} === FALLO: no está en el catálogo con ese nombre exacto")
        continue
    s = por_nombre[nombre]
    print(f"=== {nombre} ({s['state']}) ===")
    try:
        ruta_tmp = descargar_y_extraer_epw(s["url"])
        ruta_final = os.path.join(CARPETA_EPW_REAL, os.path.basename(ruta_tmp))
        shutil.copy(ruta_tmp, ruta_final)

        df, meta = cargar_epw_real(ruta_final)
        dist_vs_catalogo_km = None
        try:
            from engine.epw_real import _haversine_km
            dist_vs_catalogo_km = _haversine_km(meta["lat"], meta["lon"], s["lat"], s["lon"])
        except Exception:
            pass

        print(f"  OK -- {os.path.basename(ruta_final)}")
        print(f"  Media real: {df['WS10M'].mean():.3f} m/s | elevación: {meta['elevacion_m']:.0f} m")
        print(f"  Coordenada REAL (del EPW): lat={meta['lat']:.4f}, lon={meta['lon']:.4f}")
        print(f"  Coordenada del catálogo:   lat={s['lat']:.4f}, lon={s['lon']:.4f}"
              + (f"  -- {dist_vs_catalogo_km:.1f} km de diferencia" if dist_vs_catalogo_km else ""))

        estaciones.append(dict(nombre=nombre, estado=s["state"], archivo=os.path.basename(ruta_final),
                                media_m_s=float(df["WS10M"].mean()), elevacion_m=meta["elevacion_m"],
                                lat=meta["lat"], lon=meta["lon"]))
    except Exception as exc:
        print(f"  FALLO: {exc!r}")
    print()

print(f"Descargadas con éxito: {len(estaciones)}/{len(NOMBRES_ESTACIONES)}")

=== Dodge City Rgnl AP (KS) ===
  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/USA_United_States_of_America/KS_Kansas/USA_KS_Dodge.City.Rgnl.AP.724510_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Corpus Christi Intl AP (TX) ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/USA_United_States_of_America/TX_Texas/USA_TX_Corpus.Christi.Intl.AP.722510_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Phoenix Sky Harbor Intl AP (AZ) ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/USA_United_States_of_America/AZ_Arizona/USA_AZ_Phoenix-Sky.Harbor.Intl.AP.722780_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Reno Tahoe Intl AP (NV) ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/USA_United_States_of_America/NV_Nevada/USA_NV_Reno-Tahoe.Intl.AP.724880_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Denver Intl AP (CO) ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/USA_United_States_of_America/CO_Colorado/USA_CO_Denver.Intl.AP.725650_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Eugene AP Sweet Field (OR) ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/USA_United_States_of_America/OR_Oregon/USA_OR_Eugene.AP-Sweet.Field.726930_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Monterey Rgnl AP (CA) ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/USA_United_States_of_America/CA_California/USA_CA_Monterey.Rgnl.AP.724915_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

=== Chicago OHare Intl AP (IL) ===


  FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='climate.onebuilding.org', port=443): Max retries exceeded with url: /WMO_Region_4_North_and_Central_America/USA_United_States_of_America/IL_Illinois/USA_IL_Chicago.OHare.Intl.AP.725300_TMYx.2009-2023.zip (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))

Descargadas con éxito: 0/8


## Parte 2 — Descargar el ráster de GWA para EEUU (10m) — con chequeo de tamaño antes de seguir

In [4]:
try:
    ruta_raster_usa = descargar_raster_pais("USA", altura=10)
    tam_mb = os.path.getsize(ruta_raster_usa) / 1e6
    print(f"Ráster descargado: {ruta_raster_usa} ({tam_mb:.1f} MB)")
    if tam_mb > 2000:
        print("AVISO: más de 2 GB -- considerar si vale la pena seguir con más alturas, "
              "o recortar a menos estaciones/una región más chica antes de continuar.")
except Exception as exc:
    print(f"FALLO: {exc!r}")
    ruta_raster_usa = None

Ráster descargado: /content/ECO-Wind/datos_clima/gwa_usa_10m.tif (843.7 MB)


## Parte 3 — Diagnóstico crudo: ráster GWA (10m) vs. media real conocida, en cada estación

Mismo método que el diagnóstico crudo de Costa Rica (Hallazgo 26) -- sin ningún ajuste todavía,
solo comparar el valor crudo del ráster contra la media real ya conocida, en la coordenada REAL
del EPW (no la del catálogo).

In [5]:
try:
    ruta_raster_usa = descargar_raster_pais("USA", altura=10)
    print(f"Ráster listo: {ruta_raster_usa}\n")
except Exception as exc:
    print(f"Sin ráster descargado -- no se puede correr el diagnóstico: {exc!r}")
    ruta_raster_usa = None

if ruta_raster_usa is not None:
    filas = []
    for e in estaciones:
        try:
            media_gwa = muestrear_velocidad_media(e["lat"], e["lon"], ruta_raster_usa)
            diff_pct = (media_gwa / e["media_m_s"] - 1) * 100
            filas.append(dict(estacion=f"{e['nombre']} ({e['estado']})", media_real_m_s=e["media_m_s"],
                               media_gwa_10m=media_gwa, diferencia_pct=diff_pct,
                               elevacion_m=e["elevacion_m"]))
        except Exception as exc:
            filas.append(dict(estacion=f"{e['nombre']} ({e['estado']})", media_real_m_s=e["media_m_s"],
                               media_gwa_10m=None, diferencia_pct=f"FALLO: {exc!r}",
                               elevacion_m=e["elevacion_m"]))
    resultado = pd.DataFrame(filas)
    print(resultado.to_string(index=False))

Sin ráster descargado -- no se puede correr el diagnóstico: ProxyError(MaxRetryError("HTTPSConnectionPool(host='globalwindatlas.info', port=443): Max retries exceeded with url: /api/gis/country/USA/wind-speed/10 (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))


## Estado honesto

- Parte 1 (descarga de estaciones): sin correr todavía -- pendiente de ejecutar en Colab.
- Parte 2 (ráster de EEUU): **confirmado real -- 843.7 MB a 10m**, bien manejable (muy por debajo
  del umbral de 2 GB). `descargar_raster_pais()` ahora salta la descarga si el archivo ya existe en
  disco (`forzar=False` por defecto) -- si el runtime de Colab se reinicia entre celdas o se pierde
  la variable en memoria, no hace falta re-bajar 843 MB de nuevo, solo volver a llamar la función.
- Parte 3 (diagnóstico crudo): ahora se auto-asegura el ráster (llama a `descargar_raster_pais()` de
  nuevo al principio, gratis si ya está en disco) en vez de depender de que la variable de la Parte
  2 haya sobrevivido en la misma sesión de kernel -- corrige el `NameError` real que apareció al
  correr las partes por separado.
- Todavía no se decidió qué hacer con el patrón una vez que haya números reales -- eso es el
  siguiente paso, no parte de este notebook todavía. Si aparece una correlación clara con
  elevación/complejidad de terreno, el paso lógico siguiente sería correlacionarlo con una métrica
  de terreno real (TPI, land cover) en vez de solo mirarlo a ojo -- conecta con lo que ya propone el
  informe externo de Pablo (Hallazgo 28).
- No conectado a `app.py` -- sigue siendo investigación.